In [ ]:
"""Original AutoGluon WACC experiment (in-sample model-fit reporting).

Environment variable WACC_REGION chooses China, Europe, or Middle_EAST.
The default Middle_EAST reproduces the region used in the archived notebook.
Use a clean outputs/ directory for each training run; this code is NOT run
by the lightweight, deterministic diagnostic CI workflow.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

DATA_DIR = next(
    (p for p in (Path.cwd() / "data", Path.cwd().parent / "data")
     if (p / "Middle_EAST_WACC.xlsx").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/Middle_EAST_WACC.xlsx; run from the repository root or code/."
    )
REPO_ROOT = DATA_DIR.parent
REGION_FILES = {
    "China": "China_WACC.xlsx",
    "Europe": "Euope_WACC.xlsx",
    "Middle_EAST": "Middle_EAST_WACC.xlsx",
}
region = os.environ.get("WACC_REGION", "Middle_EAST")
if region not in REGION_FILES:
    raise ValueError(f"Invalid WACC_REGION={region!r}; choose one of {list(REGION_FILES)}.")
time_limit = int(os.environ.get("WACC_TRAINING_TIME_LIMIT", "600"))
if time_limit <= 0:
    raise ValueError("WACC_TRAINING_TIME_LIMIT must be a positive number of seconds.")
output_root = REPO_ROOT / "outputs" / "autogluon"

wacc_data = pd.read_excel(DATA_DIR / REGION_FILES[region])
wacc_data.columns = wacc_data.columns.astype(str).str.strip()
wacc_data["Month"] = pd.to_datetime(wacc_data["Month"], errors="raise")
wacc_data = wacc_data.sort_values("Month").reset_index(drop=True)
if wacc_data["Month"].duplicated().any():
    raise ValueError("Duplicate months in selected region workbook.")

projects = ["Onshore", "Offshore", "Nuclear", "CCS", "Coal", "Gas", "Solar", "Hydro", "Biomass"]
energy_types = ["Coal Price", "Gas Price", "Oil Price"]
required = projects + energy_types + ["inflation rate"]
missing = [key for key in required if key not in wacc_data.columns]
if missing:
    raise ValueError(f"Region workbook missing necessary columns: {missing}")

# Preserve the original in-sample training design and feature scaling.
scaler = MinMaxScaler()
wacc_data[energy_types] = scaler.fit_transform(wacc_data[energy_types])
results = []

for energy_type in energy_types:
    for project in projects:
        target = project
        train_data = wacc_data[[energy_type, "inflation rate", target]].dropna()
        if len(train_data) < 20:
            raise ValueError(f"Not enough training observations for {region}/{energy_type}/{project}.")
        if not np.isfinite(train_data.to_numpy(dtype=float)).all():
            raise ValueError(f"Non-finite modeling data in {region}/{energy_type}/{project}.")

        # AutoGluon model directories must be unique; sharing a single path can
        # overwrite or mix fitted models from unrelated project/fuel tasks.
        fuel_slug = energy_type.lower().replace(" ", "_")
        output_dir = output_root / "models" / region / fuel_slug / project.lower()
        if output_dir.exists() and any(output_dir.iterdir()):
            raise FileExistsError(
                f"Model output already exists: {output_dir}. "
                "Use a clean output directory instead of overwriting archived models."
            )
        output_dir.parent.mkdir(parents=True, exist_ok=True)

        predictor = TabularPredictor(label=target, path=str(output_dir)).fit(
            train_data,
            time_limit=time_limit,
            presets="best_quality",
            hyperparameters={
                "GBM": {
                    "max_depth": 3,
                    "learning_rate": 0.01,
                    "num_boost_round": 100,
                    "min_child_weight": 5,
                },
                "CAT": {
                    "depth": 3,
                    "learning_rate": 0.01,
                    "iterations": 100,
                    "l2_leaf_reg": 20,
                },
                "NN_TORCH": {
                    "num_layers": 1,
                    "hidden_size": 16,
                    "dropout": 0.3,
                },
            },
            num_bag_folds=10,
            keep_only_best=True,
            auto_stack=True,
        )

        performance = predictor.evaluate(train_data)
        predictions = predictor.predict(train_data)
        mape = mean_absolute_percentage_error(train_data[target], predictions)
        rmse = np.sqrt(mean_squared_error(train_data[target], predictions))
        results.append({
            "Region": region,
            "Energy Type": energy_type,
            "Project": project,
            "Evaluation": "in_sample",
            "R²": performance["r2"],
            "MAPE": mape,
            "RMSE": rmse,
        })

results_df = pd.DataFrame(results)
table_dir = output_root / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
result_path = table_dir / f"{region}_results.csv"
results_df.to_csv(result_path, index=False)
print(f"Saved {len(results_df)} in-sample fits to {result_path}")
print(results_df)
